# M1 Notebook 18 — Hypothesis Testing and Effect Size

**Notebook ID:** M1_N18  
**Status:** Runnable first edition  
**Random seed:** 42

> Hypothesis tests quantify compatibility with a null model. Effect sizes quantify the magnitude of a difference. Good inference requires both.


## 1. Learning objectives

1. Define null and alternative hypotheses.
2. Interpret test statistics and p-values.
3. Perform one-sample, independent, paired, and proportion tests.
4. Distinguish statistical significance from practical importance.
5. Compute standardized effect sizes.
6. Understand Type I error, Type II error, and power.
7. Apply testing responsibly in AI and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.statistics import (
    cohen_d_independent,
    cohen_d_one_sample,
    cohen_d_paired,
    one_sample_t_test,
    paired_t_test,
    power_one_sample_z,
    proportion_z_test,
    two_sample_t_test,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## 2. Hypotheses

A null hypothesis specifies a reference model, for example

\[
H_0:\mu=\mu_0.
\]

The alternative may be two-sided,

\[
H_1:\mu\ne\mu_0,
\]

or directional.


## 3. Test statistic and p-value

A test statistic measures disagreement between the data and the null model.

The p-value is the probability, under the null model, of obtaining a result at least as extreme as the observed result.

It is not the probability that the null hypothesis is true.


## 4. One-sample t test

In [ ]:
sample = np.array([12.1, 11.8, 12.5, 12.3, 11.9, 12.6, 12.4, 12.2])

t_stat, p_value = one_sample_t_test(
    sample,
    null_mean=12.0,
)

effect = cohen_d_one_sample(
    sample,
    null_mean=12.0,
)

{
    "sample_mean": sample.mean(),
    "t_statistic": t_stat,
    "p_value": p_value,
    "Cohen_d": effect,
}


## 5. Independent two-sample t test

In [ ]:
rng = np.random.default_rng(42)

group_a = rng.normal(105, 12, 80)
group_b = rng.normal(100, 12, 80)

t_stat, p_value = two_sample_t_test(
    group_a,
    group_b,
    equal_var=False,
)

d = cohen_d_independent(
    group_a,
    group_b,
)

{
    "mean_A": group_a.mean(),
    "mean_B": group_b.mean(),
    "mean_difference": group_a.mean() - group_b.mean(),
    "t_statistic": t_stat,
    "p_value": p_value,
    "Cohen_d": d,
}


Welch's t test is generally preferred when equal variances cannot be justified.


## 6. Paired t test

In [ ]:
before = np.array([70, 68, 75, 72, 80, 77, 74, 69], dtype=float)
after = np.array([74, 70, 79, 75, 84, 80, 78, 73], dtype=float)

t_stat, p_value = paired_t_test(before, after)
paired_effect = cohen_d_paired(before, after)

{
    "mean_change": np.mean(after - before),
    "t_statistic": t_stat,
    "p_value": p_value,
    "paired_Cohen_d": paired_effect,
}


Paired tests analyze within-unit differences and should not be replaced by an independent-groups test.


## 7. One-sample proportion test

In [ ]:
successes = 62
trials = 100

z_stat, p_value = proportion_z_test(
    successes,
    trials,
    null_proportion=0.50,
)

{
    "sample_proportion": successes/trials,
    "z_statistic": z_stat,
    "p_value": p_value,
}


## 8. Statistical significance versus effect size

In [ ]:
sample_sizes = [20, 100, 1000, 10000]
true_effect = 0.20
rows = []

for n in sample_sizes:
    rng_local = np.random.default_rng(100 + n)
    x = rng_local.normal(true_effect, 1.0, n)
    t_stat, p_value = one_sample_t_test(x, 0.0)
    d = cohen_d_one_sample(x, 0.0)
    rows.append({
        "sample_size": n,
        "sample_mean": x.mean(),
        "p_value": p_value,
        "Cohen_d": d,
    })

pd.DataFrame(rows)


A very small effect can become statistically significant with a large sample. Practical relevance requires domain-specific interpretation.


## 9. Visualizing null and alternative distributions

In [ ]:
alpha = 0.05
critical = stats.norm.ppf(1 - alpha/2)
x = np.linspace(-4, 5, 800)

null_density = stats.norm.pdf(x, loc=0, scale=1)
alternative_density = stats.norm.pdf(x, loc=1.5, scale=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, null_density, label="Null distribution")
ax.plot(x, alternative_density, label="Alternative distribution")
ax.axvline(-critical, linestyle="--")
ax.axvline(critical, linestyle="--")
ax.set_xlabel("Test statistic")
ax.set_ylabel("Density")
ax.set_title("Null, Alternative, and Rejection Regions")
ax.legend()
plt.show()


## 10. Type I and Type II errors

- Type I error: reject \(H_0\) when it is true.
- Type II error: fail to reject \(H_0\) when a meaningful alternative is true.
- Power: probability of rejecting \(H_0\) when the alternative is true.


## 11. Power and sample size

In [ ]:
sample_sizes = np.arange(10, 301, 10)
effect_sizes = [0.2, 0.5, 0.8]

fig, ax = plt.subplots(figsize=(8, 5))
for effect_size in effect_sizes:
    powers = [
        power_one_sample_z(
            effect_size,
            int(n),
            alpha=0.05,
            alternative="two-sided",
        )
        for n in sample_sizes
    ]
    ax.plot(sample_sizes, powers, label=f"d={effect_size}")

ax.axhline(0.8, linestyle="--")
ax.set_xlabel("Sample size")
ax.set_ylabel("Power")
ax.set_title("Power Curves")
ax.legend()
plt.show()


Power rises with sample size and effect magnitude.


## 12. Simulating Type I error

In [ ]:
rng = np.random.default_rng(7)
repetitions = 5000
rejections = 0

for _ in range(repetitions):
    x = rng.normal(0.0, 1.0, 30)
    _, p = one_sample_t_test(x, 0.0)
    rejections += p < 0.05

empirical_type_i = rejections / repetitions
empirical_type_i


Under valid assumptions, the long-run rejection rate should be close to the chosen significance level.


## 13. Multiple testing problem

In [ ]:
rng = np.random.default_rng(123)
experiments = 1000
tests_per_experiment = 20
at_least_one_false_positive = []

for _ in range(experiments):
    p_values = []
    for _ in range(tests_per_experiment):
        x = rng.normal(0, 1, 30)
        _, p = one_sample_t_test(x, 0.0)
        p_values.append(p)
    at_least_one_false_positive.append(
        np.any(np.array(p_values) < 0.05)
    )

{
    "empirical_familywise_false_positive_rate": np.mean(
        at_least_one_false_positive
    ),
    "theoretical_independent_rate": 1 - (1 - 0.05)**tests_per_experiment,
}


Repeated unadjusted testing increases false-positive risk.


## 14. Statistics interpretation

Hypothesis testing is a decision procedure under a reference model. It should be accompanied by estimates, confidence intervals, effect sizes, assumptions, and study design.


## 15. AI interpretation

Testing supports:

- A/B experiments;
- model comparisons;
- fairness audits;
- drift detection;
- benchmark evaluation;
- feature-ablation studies.

Dependence, repeated evaluation, and test-set reuse can invalidate naive p-values.


## 16. Decision Intelligence case — Programme evaluation

Suppose an intervention is designed to reduce average processing time.


In [ ]:
rng = np.random.default_rng(222)
control = rng.normal(15.0, 3.0, 120)
intervention = rng.normal(13.8, 3.0, 120)

t_stat, p_value = two_sample_t_test(
    intervention,
    control,
    equal_var=False,
    alternative="less",
)

effect_size = cohen_d_independent(
    intervention,
    control,
)

decision_summary = {
    "control_mean": control.mean(),
    "intervention_mean": intervention.mean(),
    "estimated_reduction": control.mean() - intervention.mean(),
    "one_sided_p_value": p_value,
    "Cohen_d_intervention_minus_control": effect_size,
}
decision_summary


### Interpretation

The p-value addresses compatibility with a null model. The estimated reduction and effect size address magnitude. A policy recommendation also requires cost, implementation quality, external validity, equity, and operational significance.


## 17. Engineering notes

- Test validity depends on design and assumptions.
- Large samples can make negligible effects significant.
- Small samples may have low power.
- Multiple testing requires adjustment or hierarchical planning.
- Optional stopping can inflate false positives.
- Pre-registration and analysis plans reduce researcher degrees of freedom.


## 18. Common errors

- Interpreting \(p<0.05\) as proof that a hypothesis is true.
- Treating non-significance as evidence of no effect.
- Reporting p-values without effect sizes.
- Choosing one-sided tests after seeing the data.
- Ignoring multiple comparisons.
- Confusing statistical and practical significance.


## 19. Exercises

### Level A
Define p-value, Type I error, Type II error, and power.

### Level B
Derive the one-sample t statistic.

### Level C
Simulate power curves and familywise error under multiple testing.

### Capstone
Evaluate a programme with an appropriate design, estimate effect size, perform a justified hypothesis test, assess power, and discuss practical and ethical significance.


## 20. Key insight

Hypothesis tests quantify evidence against a null model. Effect sizes quantify magnitude. Responsible inference combines both with confidence intervals, power, assumptions, study design, and substantive judgment.
